# SLM-RAG Step 1: fix the measurement

Runs on **Kaggle** (Accelerator: **GPU T4 x2 or P100**, Internet: **On**). Nothing needs uploading: every module is written to disk by the cells below.

What this notebook does
1. installs dependencies and writes the evaluation package (`slmrag_*.py`)
2. runs the 34 unit tests and checks the dataset schemas
3. (optional) diagnoses your **old** predictions to confirm the F1 floor is a verbosity artefact
4. smoke test (20 questions), then the baseline matrix: closed-book / oracle / BM25 / dense / hybrid
5. paired statistics with Holm correction, a summary table, and a zip of all outputs

**Gate to pass before Step 2:** no WARNING lines in the smoke test, and oracle beats closed-book (paired CI excludes 0) on HotpotQA and PubMedQA.

## 0. Environment

In [1]:
import os, sys, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one 16 GB GPU, as in the paper (set BEFORE torch loads CUDA)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.chdir("/kaggle/working")

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn on a GPU: Settings > Accelerator"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

torch 2.11.0+cu128 | cuda available: True
Tesla T4 15.6 GB


In [2]:
# bitsandbytes is only needed for --load_4bit; the rest is normally pre-installed on Kaggle.
!pip install -q bitsandbytes sentence-transformers datasets accelerate scipy pandas pytest
# If dataset loading fails later with a 'scripts are no longer supported' error, run:  !pip install -q "datasets<4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 28.6 MB/s eta 0:00:0000:0100:01m


## 1. Write the package to disk

In [3]:
%%writefile slmrag_core.py
"""slmrag_core: measurement-fixed evaluation core for the SLM-RAG study (Step 1).

Dependency-light on purpose (numpy + scipy only) so every function here is
unit-tested without a GPU. Model code lives in slmrag_hf.py, dataset code in
slmrag_data.py, and both plug into `evaluate()` below through plain callables.

Sections
  1. Answer normalisation and metrics (EM, F1, containment accuracy, PubMedQA labels)
  2. Generation post-processing (stop handling, answer extraction)
  3. Prompts and context budgeting
  4. Retrieval (BM25, dense, hybrid RRF) and retrieval metrics
  5. Statistics (bootstrap CI, paired tests, Holm, McNemar, power, Kendall tau)
  6. evaluate(): one setting, one model, per-example records
"""
from __future__ import annotations

import collections
import math
import re
import string
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
from scipy import sparse, stats as sps

# --------------------------------------------------------------------------
# 1. Normalisation and metrics
# --------------------------------------------------------------------------
_PUNCT = set(string.punctuation)
_ARTICLES = re.compile(r"\b(a|an|the)\b", re.I)


def normalize_answer(s: str) -> str:
    """SQuAD-style normalisation: lowercase, strip punctuation and articles."""
    s = s.lower()
    s = "".join(ch for ch in s if ch not in _PUNCT)
    s = _ARTICLES.sub(" ", s)
    return " ".join(s.split())


def _as_list(golds) -> List[str]:
    return [golds] if isinstance(golds, str) else list(golds)


def exact_match(pred: str, golds) -> float:
    p = normalize_answer(pred)
    return float(any(p == normalize_answer(g) for g in _as_list(golds)))


def _f1_single(pred: str, gold: str) -> float:
    pt, gt = normalize_answer(pred).split(), normalize_answer(gold).split()
    if not pt or not gt:
        return float(pt == gt)
    common = collections.Counter(pt) & collections.Counter(gt)
    n = sum(common.values())
    if n == 0:
        return 0.0
    prec, rec = n / len(pt), n / len(gt)
    return 2 * prec * rec / (prec + rec)


def f1_score(pred: str, golds) -> float:
    """Token F1, maximum over gold aliases."""
    return max(_f1_single(pred, g) for g in _as_list(golds))


def contains_answer(text: str, golds) -> bool:
    """True if any normalised gold answer occurs in `text` on token boundaries."""
    t = f" {normalize_answer(text)} "
    for g in _as_list(golds):
        ng = normalize_answer(g)
        if ng and f" {ng} " in t:
            return True
    return False


def acc_contain(pred: str, golds) -> float:
    """Containment accuracy (standard for NQ-Open / PopQA style evaluation)."""
    return float(contains_answer(pred, golds))


_LABEL_RE = re.compile(r"\b(yes|no|maybe)\b", re.I)
LABELS = ("yes", "no", "maybe")


def extract_label(pred: str) -> Optional[str]:
    """First yes/no/maybe word in the (cleaned) prediction, else None."""
    m = _LABEL_RE.search(pred or "")
    return m.group(1).lower() if m else None


def label_metrics(preds: Sequence[Optional[str]], golds: Sequence[str]) -> Dict[str, float]:
    """Accuracy and macro-F1 for PubMedQA decisions. Unparsable predictions count as wrong."""
    assert len(preds) == len(golds)
    acc = float(np.mean([p == g for p, g in zip(preds, golds)])) if golds else 0.0
    f1s = []
    for lab in LABELS:
        tp = sum(p == lab and g == lab for p, g in zip(preds, golds))
        fp = sum(p == lab and g != lab for p, g in zip(preds, golds))
        fn = sum(p != lab and g == lab for p, g in zip(preds, golds))
        f1s.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return {"accuracy": acc, "macro_f1": float(np.mean(f1s))}


# --------------------------------------------------------------------------
# 2. Generation post-processing
# --------------------------------------------------------------------------
_STOP = re.compile(
    r"(\n|\bQuestion\s*:|\bQ\s*:|\bContext\s*:|\bPassages?\b\s*[\[:(]|\bExplanation\s*:|"
    r"<\|im_end\|>|<\|endoftext\|>|<\|eot_id\|>|###)",
    re.I,
)
_PREFIX = re.compile(r"^\s*(final\s+answer|answer|a)\s*[:\-]\s*", re.I)


def clean_generation(raw: str, max_words: Optional[int] = None) -> str:
    """Reduce a raw generation to the answer span.

    Cuts at the first newline or at a hallucinated follow-up ("Question:", "Context:" ...),
    removes "Answer:" prefixes, surrounding quotes and a trailing full stop.
    """
    s = (raw or "").strip()
    m = _STOP.search(s)
    if m:
        s = s[: m.start()]
    prev = None
    while prev != s:  # strip repeated prefixes
        prev, s = s, _PREFIX.sub("", s).strip()
    s = s.strip().strip("\"'`*").strip()
    if s.endswith(".") and not re.search(r"\b[A-Z]\.$", s):
        s = s[:-1].strip()
    if max_words is not None:
        s = " ".join(s.split()[:max_words])
    return s


# --------------------------------------------------------------------------
# 3. Prompts and context budgeting
# --------------------------------------------------------------------------
SYSTEM_SHORT = (
    "You are a precise question-answering assistant. Reply with only the answer, "
    "as a short phrase of at most five words. Do not explain. Do not repeat the question."
)
SYSTEM_YESNO = (
    "You are a precise biomedical question-answering assistant. Reply with exactly one word: "
    "yes, no, or maybe. Do not explain."
)


def format_passages(passages: Sequence[str]) -> str:
    return "\n".join(f"[{i + 1}] {p.strip()}" for i, p in enumerate(passages))


def truncate_passages(
    passages: Sequence[str],
    budget: int,
    len_fn: Callable[[str], int] = lambda s: len(s.split()),
) -> List[str]:
    """Keep whole passages in rank order until `budget` is spent.

    If even the first passage exceeds the budget it is cut by words. The same
    budget must be used for every retriever so context length is not a confound.
    """
    out, used = [], 0
    for p in passages:
        n = len_fn(p)
        if used + n <= budget:
            out.append(p)
            used += n
        elif not out:
            out.append(" ".join(p.split()[:budget]))
            break
        else:
            break
    return out


def build_messages(question: str, passages: Optional[Sequence[str]], task: str = "qa") -> List[dict]:
    """Chat messages. passages=None means closed-book."""
    system = SYSTEM_YESNO if task == "yesno" else SYSTEM_SHORT
    if passages:
        user = f"Context:\n{format_passages(passages)}\n\nQuestion: {question.strip()}\nAnswer:"
    else:
        user = f"Question: {question.strip()}\nAnswer:"
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


# --------------------------------------------------------------------------
# 4. Retrieval
# --------------------------------------------------------------------------
_STOPWORDS = frozenset(
    "a an the of in on at to for from by with and or is are was were be been it its this that "
    "as which who whom what when where how do does did".split()
)
_TOK = re.compile(r"\w+")


def tokenize(text: str) -> List[str]:
    return [t for t in _TOK.findall(text.lower()) if t not in _STOPWORDS]


def chunk_words(text: str, size: int = 128, overlap: int = 0) -> List[str]:
    w = text.split()
    if not w:
        return []
    step = max(1, size - overlap)
    return [" ".join(w[i : i + size]) for i in range(0, max(1, len(w) - overlap), step)]


class BM25:
    """Okapi BM25 (Anserini defaults k1=0.9, b=0.4) on a precomputed sparse weight matrix."""

    def __init__(self, docs: Sequence[str], k1: float = 0.9, b: float = 0.4):
        self.n = len(docs)
        vocab: Dict[str, int] = {}
        rows, cols, tfs, dl = [], [], [], np.zeros(self.n)
        for i, d in enumerate(docs):
            toks = tokenize(d)
            dl[i] = len(toks)
            for t, c in collections.Counter(toks).items():
                rows.append(i)
                cols.append(vocab.setdefault(t, len(vocab)))
                tfs.append(c)
        self.vocab = vocab
        rows, cols, tfs = np.array(rows), np.array(cols), np.array(tfs, dtype=float)
        df = np.bincount(cols, minlength=len(vocab)).astype(float)
        idf = np.log(1.0 + (self.n - df + 0.5) / (df + 0.5))
        avgdl = max(dl.mean(), 1e-9)
        w = idf[cols] * tfs * (k1 + 1) / (tfs + k1 * (1 - b + b * dl[rows] / avgdl))
        self.W = sparse.csc_matrix((w, (rows, cols)), shape=(self.n, len(vocab)))

    def scores(self, query: str) -> np.ndarray:
        idx = sorted({self.vocab[t] for t in tokenize(query) if t in self.vocab})
        if not idx:
            return np.zeros(self.n)
        return np.asarray(self.W[:, idx].sum(axis=1)).ravel()

    def search(self, query: str, k: int = 10) -> List[Tuple[int, float]]:
        s = self.scores(query)
        k = min(k, self.n)
        top = np.argpartition(-s, k - 1)[:k]
        top = top[np.argsort(-s[top], kind="stable")]
        return [(int(i), float(s[i])) for i in top]


class DenseRetriever:
    """Exact inner-product search over L2-normalised embeddings (numpy, no FAISS needed).

    encode_fn(list[str]) -> float array [n, d]. Use make_st_encoder() for sentence-transformers.
    """

    def __init__(self, docs: Sequence[str], encode_fn: Callable[[List[str]], np.ndarray],
                 query_prefix: str = "", doc_prefix: str = "", batch: int = 64):
        self.encode, self.qp = encode_fn, query_prefix
        embs = [self._norm(encode_fn([doc_prefix + d for d in docs[i : i + batch]]))
                for i in range(0, len(docs), batch)]
        self.E = np.vstack(embs).astype(np.float32)

    @staticmethod
    def _norm(x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, dtype=np.float32)
        return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-9)

    def search(self, query: str, k: int = 10) -> List[Tuple[int, float]]:
        q = self._norm(self.encode([self.qp + query]))[0]
        s = self.E @ q
        k = min(k, len(s))
        top = np.argpartition(-s, k - 1)[:k]
        top = top[np.argsort(-s[top], kind="stable")]
        return [(int(i), float(s[i])) for i in top]


def make_st_encoder(model_name: str = "intfloat/e5-base-v2", device: Optional[str] = None):
    """sentence-transformers encoder. E5 needs prefixes: pass query_prefix='query: ', doc_prefix='passage: '."""
    from sentence_transformers import SentenceTransformer  # lazy: heavy import

    m = SentenceTransformer(model_name, device=device)
    return lambda texts: m.encode(texts, convert_to_numpy=True, normalize_embeddings=True,
                                  show_progress_bar=False)


def rrf(rank_lists: Sequence[Sequence[int]], k: int = 60, top: int = 10) -> List[Tuple[int, float]]:
    """Reciprocal rank fusion over several ranked lists of doc ids."""
    score: Dict[int, float] = collections.defaultdict(float)
    for lst in rank_lists:
        for r, d in enumerate(lst):
            score[d] += 1.0 / (k + r + 1)
    return sorted(score.items(), key=lambda kv: (-kv[1], kv[0]))[:top]


class HybridRRF:
    def __init__(self, a, b, depth: int = 100, k: int = 60):
        self.a, self.b, self.depth, self.k = a, b, depth, k

    def search(self, query: str, k: int = 10) -> List[Tuple[int, float]]:
        la = [d for d, _ in self.a.search(query, self.depth)]
        lb = [d for d, _ in self.b.search(query, self.depth)]
        return rrf([la, lb], k=self.k, top=k)


def retrieval_metrics(retrieved_texts: Sequence[Sequence[str]], golds: Sequence,
                      ks: Sequence[int] = (1, 5, 10)) -> Dict[str, float]:
    """Answer-containment retrieval quality: hit@k and MRR@max(k). Queries with no answer string are skipped."""
    hits = {k: [] for k in ks}
    rr = []
    kmax = max(ks)
    for texts, g in zip(retrieved_texts, golds):
        flags = [contains_answer(t, g) for t in texts[:kmax]]
        for k in ks:
            hits[k].append(any(flags[:k]))
        rr.append(next((1.0 / (i + 1) for i, f in enumerate(flags) if f), 0.0))
    out = {f"hit@{k}": float(np.mean(v)) if v else 0.0 for k, v in hits.items()}
    out[f"mrr@{kmax}"] = float(np.mean(rr)) if rr else 0.0
    return out


def id_recall_at_k(retrieved_ids: Sequence[Sequence], gold_ids: Sequence[Sequence], k: int) -> float:
    """Mean fraction of gold documents found in the top-k (queries without gold ids are skipped)."""
    vals = [len(set(r[:k]) & set(g)) / len(set(g)) for r, g in zip(retrieved_ids, gold_ids) if g]
    return float(np.mean(vals)) if vals else float("nan")


# --------------------------------------------------------------------------
# 5. Statistics
# --------------------------------------------------------------------------
def bootstrap_ci(x, B: int = 10000, alpha: float = 0.05, seed: int = 0) -> Tuple[float, float, float]:
    """(mean, lo, hi) percentile bootstrap, chunked to bound memory."""
    x = np.asarray(x, dtype=float)
    n, rng = len(x), np.random.default_rng(seed)
    means = np.empty(B)
    step = max(1, int(2e7 // max(n, 1)))
    for s in range(0, B, step):
        e = min(B, s + step)
        means[s:e] = x[rng.integers(0, n, (e - s, n))].mean(axis=1)
    lo, hi = np.quantile(means, [alpha / 2, 1 - alpha / 2])
    return float(x.mean()), float(lo), float(hi)


def paired_compare(a, b, binary: bool = False, B: int = 10000, seed: int = 0) -> Dict[str, float]:
    """Paired comparison of per-example scores a vs b (same examples, same order).

    Returns mean difference (a-b), bootstrap CI, paired t, Wilcoxon, Cohen's dz and,
    for 0/1 scores, the exact McNemar p-value (the correct test for accuracy).
    """
    a, b = np.asarray(a, float), np.asarray(b, float)
    assert a.shape == b.shape, "paired arrays must align"
    d = a - b
    mean, lo, hi = bootstrap_ci(d, B=B, seed=seed)
    sd = d.std(ddof=1) if len(d) > 1 else 0.0
    out = {"n": len(d), "mean_diff": mean, "ci_lo": lo, "ci_hi": hi,
           "dz": float(mean / sd) if sd > 0 else 0.0, "sd_diff": float(sd)}
    out["p_t"] = float(sps.ttest_rel(a, b).pvalue) if sd > 0 else 1.0
    nz = d[d != 0]
    out["p_wilcoxon"] = float(sps.wilcoxon(nz).pvalue) if len(nz) > 0 else 1.0
    if binary:
        n01, n10 = int(((a == 0) & (b == 1)).sum()), int(((a == 1) & (b == 0)).sum())
        out["p_mcnemar"] = float(sps.binomtest(n10, n10 + n01, 0.5).pvalue) if n10 + n01 > 0 else 1.0
        out["discordant"] = n10 + n01
    return out


def holm(pvals: Sequence[float]) -> List[float]:
    """Holm-Bonferroni adjusted p-values (monotone, capped at 1)."""
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj, running = np.empty_like(p), 0.0
    m = len(p)
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj.tolist()


def mde_paired(n: int, sd_diff: float, alpha: float = 0.05, power: float = 0.8) -> float:
    """Minimum detectable mean difference for a paired test (normal approximation)."""
    z = sps.norm.ppf(1 - alpha / 2) + sps.norm.ppf(power)
    return float(z * sd_diff / math.sqrt(n))


def n_required(delta: float, sd_diff: float, alpha: float = 0.05, power: float = 0.8) -> int:
    z = sps.norm.ppf(1 - alpha / 2) + sps.norm.ppf(power)
    return int(math.ceil((z * sd_diff / delta) ** 2))


def kendall_tau(x: Sequence[float], y: Sequence[float]) -> Tuple[float, float]:
    r = sps.kendalltau(x, y)
    return float(r.statistic), float(r.pvalue)


# --------------------------------------------------------------------------
# 6. evaluate(): one setting, one model
# --------------------------------------------------------------------------
def evaluate(
    records: Sequence[dict],
    generate_fn: Callable[[List[List[dict]]], List[Tuple[str, bool]]],
    retrieve_fn: Optional[Callable[[dict], List[str]]] = None,
    context_budget: int = 600,
    len_fn: Callable[[str], int] = lambda s: len(s.split()),
    max_words: Optional[int] = None,
) -> Tuple[List[dict], Dict[str, float]]:
    """Run generation for every record and score it.

    records: dicts with id, question, answers (list[str]), task ('qa'|'yesno'), label (yesno only)
    generate_fn: list of chat-message lists -> list of (raw_text, hit_max_new_tokens)
    retrieve_fn: record -> ranked passage strings (None = closed-book)
    Returns (per-example rows, summary). Rows keep the RAW output so any post-processing
    can be audited later.
    """
    prompts, ctxs = [], []
    for r in records:
        ps = retrieve_fn(r) if retrieve_fn else None
        ps = truncate_passages(ps, context_budget, len_fn) if ps else None
        ctxs.append(ps)
        prompts.append(build_messages(r["question"], ps, r.get("task", "qa")))
    outs = generate_fn(prompts)
    assert len(outs) == len(records)

    rows = []
    for r, ps, (raw, hit_max) in zip(records, ctxs, outs):
        pred = clean_generation(raw, max_words=max_words)
        row = {"id": r["id"], "task": r.get("task", "qa"), "raw": raw, "pred": pred,
               "hit_max": bool(hit_max), "n_ctx": len(ps) if ps else 0}
        if r.get("task", "qa") == "yesno":
            lab = extract_label(pred)
            row.update(label=r["label"], pred_label=lab, acc=float(lab == r["label"]))
        else:
            row.update(answers=r["answers"], em=exact_match(pred, r["answers"]),
                       f1=f1_score(pred, r["answers"]), acc=acc_contain(pred, r["answers"]),
                       raw_f1=f1_score(raw, r["answers"]), raw_acc=acc_contain(raw, r["answers"]),
                       ctx_has_answer=bool(ps) and any(contains_answer(p, r["answers"]) for p in ps))
        rows.append(row)
    return rows, summarize(rows)


def summarize(rows: Sequence[dict]) -> Dict[str, float]:
    s: Dict[str, float] = {"n": len(rows)}
    qa = [r for r in rows if r["task"] == "qa"]
    yn = [r for r in rows if r["task"] == "yesno"]
    if qa:
        for k in ("em", "f1", "acc"):
            m, lo, hi = bootstrap_ci([r[k] for r in qa], B=2000)
            s.update({k: m, f"{k}_lo": lo, f"{k}_hi": hi})
        s["raw_f1"] = float(np.mean([r["raw_f1"] for r in qa]))
        s["mean_pred_words"] = float(np.mean([len(r["pred"].split()) for r in qa]))
        s["mean_raw_words"] = float(np.mean([len(r["raw"].split()) for r in qa]))
    if yn:
        s.update({f"yn_{k}": v for k, v in label_metrics(
            [r["pred_label"] for r in yn], [r["label"] for r in yn]).items()})
        s["yn_unparsed"] = float(np.mean([r["pred_label"] is None for r in yn]))
    s["hit_max_rate"] = float(np.mean([r["hit_max"] for r in rows])) if rows else 0.0
    s["empty_rate"] = float(np.mean([not r["pred"] for r in rows])) if rows else 0.0
    return s


def sanity_warnings(summaries: Dict[str, Dict[str, float]]) -> List[str]:
    """Objective red flags across settings (keys are setting names such as 'closed', 'oracle')."""
    w = []
    for name, s in summaries.items():
        if s.get("hit_max_rate", 0) > 0.02:
            w.append(f"[{name}] {s['hit_max_rate']:.1%} of generations hit max_new_tokens: stop handling is not working.")
        if s.get("empty_rate", 0) > 0.02:
            w.append(f"[{name}] {s['empty_rate']:.1%} empty predictions after cleaning.")
        if s.get("mean_pred_words", 0) > 8:
            w.append(f"[{name}] mean cleaned answer is {s['mean_pred_words']:.1f} words: prompt is not producing short answers.")
        if s.get("raw_f1") is not None and "f1" in s and s["f1"] > 0 and s["f1"] / max(s["raw_f1"], 1e-9) > 3:
            w.append(f"[{name}] cleaned F1 is >3x raw F1: results depend heavily on post-processing, report both.")
        if s.get("yn_unparsed", 0) > 0.05:
            w.append(f"[{name}] {s['yn_unparsed']:.1%} of PubMedQA outputs contain no yes/no/maybe.")
    if "closed" in summaries and "oracle" in summaries:
        c, o = summaries["closed"], summaries["oracle"]
        key = "acc" if "acc" in c else "yn_accuracy"
        if key in c and key in o and o[key] <= c[key]:
            w.append("oracle context does not beat closed-book: the generation or prompt path is still broken.")
    return w

Writing slmrag_core.py


In [4]:
%%writefile slmrag_data.py
"""Dataset loaders for Step 1. Pure transform functions are unit-tested; the HF download
calls are not (no network in the dev sandbox), so run `python slmrag_data.py --check` once on
your machine: it prints one record per dataset and fails loudly if a schema differs.

Record format (all datasets)
  id, question, task ('qa'|'yesno'), answers (list[str]; qa), label (yesno),
  gold_passages (list[str]), gold_ids (list[str])
"""
from __future__ import annotations

import json
import random
import sys
from typing import Dict, List, Optional, Tuple


# ---- pure transforms -------------------------------------------------------
def nq_record(i: int, row: dict) -> dict:
    q = row["question"].strip()
    if not q.endswith("?"):  # nq_open questions are lower-case fragments without a question mark
        q += "?"
    return {"id": f"nq-{i}", "question": q, "task": "qa", "answers": list(row["answer"]),
            "gold_passages": [], "gold_ids": []}


def hotpot_paragraphs(row: dict) -> List[Tuple[str, str]]:
    ctx = row["context"]
    return [(t, f"{t}: {' '.join(s).strip()}") for t, s in zip(ctx["title"], ctx["sentences"])]


def hotpot_record(row: dict) -> dict:
    paras = dict(hotpot_paragraphs(row))
    gold_titles = list(dict.fromkeys(row["supporting_facts"]["title"]))  # unique, ordered
    return {"id": f"hp-{row['id']}", "question": row["question"].strip(), "task": "qa",
            "answers": [row["answer"]], "gold_passages": [paras[t] for t in gold_titles if t in paras],
            "gold_ids": [t for t in gold_titles if t in paras]}


def pubmed_record(row: dict) -> dict:
    ctx = " ".join(row["context"]["contexts"]).strip()
    return {"id": f"pm-{row['pubid']}", "question": row["question"].strip(), "task": "yesno",
            "label": row["final_decision"].strip().lower(), "answers": [],
            "gold_passages": [ctx], "gold_ids": [str(row["pubid"])]}


# ---- loaders ---------------------------------------------------------------
def _load(candidates, **kw):
    from datasets import load_dataset  # lazy
    last = None
    for c in candidates:
        try:
            return load_dataset(*c, **kw)
        except Exception as e:  # dataset renamed / script-based loader removed, try next id
            last = e
    raise RuntimeError(f"could not load any of {candidates}: {last}")


def _sample(idx: List[int], n: Optional[int], seed: int) -> List[int]:
    idx = list(idx)
    random.Random(seed).shuffle(idx)
    return idx[:n] if n else idx


def load_nq(n: int, seed: int = 42, split: str = "validation") -> Tuple[List[dict], Optional[List[dict]]]:
    ds = _load([("google-research-datasets/nq_open",), ("nq_open",)], split=split)
    return [nq_record(i, ds[i]) for i in _sample(range(len(ds)), n, seed)], None  # no pool: needs a Wikipedia corpus (Step 2)


def load_hotpot(n: int, seed: int = 42, split: str = "validation") -> Tuple[List[dict], List[dict]]:
    """Returns (eval records, pool). Pool = every paragraph in the split (closed pool: easier than open-domain)."""
    ds = _load([("hotpotqa/hotpot_qa", "distractor"), ("hotpot_qa", "distractor")], split=split)
    pool: Dict[str, dict] = {}
    for row in ds:
        for t, text in hotpot_paragraphs(row):
            pool.setdefault(text, {"id": t, "text": text})
    recs = [hotpot_record(ds[i]) for i in _sample(range(len(ds)), n, seed)]
    return recs, list(pool.values())


def load_pubmed(n: int, seed: int = 42, n_train: int = 0) -> Tuple[List[dict], List[dict], List[dict]]:
    """pqa_labeled has 1000 rows. Returns (eval, pool, train): eval = first n of a seeded shuffle,
    train = the next n_train (kept disjoint from eval for Step 5 fine-tuning)."""
    ds = _load([("qiaojin/PubMedQA", "pqa_labeled"), ("pubmed_qa", "pqa_labeled")], split="train")
    order = _sample(range(len(ds)), None, seed)
    allrecs = [pubmed_record(ds[i]) for i in order]
    pool = [{"id": r["gold_ids"][0], "text": r["gold_passages"][0]} for r in allrecs]
    return allrecs[:n], pool, allrecs[n : n + n_train]


def load_corpus_jsonl(path: str) -> List[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


if __name__ == "__main__":
    if "--check" in sys.argv:
        nq, _ = load_nq(2)
        hp, pool = load_hotpot(2)
        pm, ppool, _ = load_pubmed(2)
        for name, r in (("nq", nq[0]), ("hotpot", hp[0]), ("pubmed", pm[0])):
            print(name, {k: (v if k != "gold_passages" else [p[:80] for p in v]) for k, v in r.items()})
        assert hp[0]["gold_passages"], "HotpotQA: supporting-fact titles did not match context titles"
        assert pm[0]["label"] in ("yes", "no", "maybe")
        print(f"hotpot pool={len(pool)} paragraphs, pubmed pool={len(ppool)} abstracts. Schemas OK.")

Writing slmrag_data.py


In [5]:
%%writefile slmrag_hf.py
"""Hugging Face generation backend for Step 1 (needs torch + transformers; not unit-tested
in the dev sandbox, which has no GPU or model access -- see PROTOCOL.md smoke test).

Key differences from the old pipeline that caused the F1 floor:
  * chat template + a short-answer system prompt
  * generation stops on the model's real end-of-turn tokens (not just the base EOS)
  * max_new_tokens=32, greedy
  * the RAW generation and a hit-the-limit flag are returned so rambling is measurable
"""
from __future__ import annotations

from typing import List, Tuple


def load_model(name: str, load_4bit: bool = False):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tok = AutoTokenizer.from_pretrained(name, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = {}
    if load_4bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    else:
        # T4 has no bf16. Gemma-2 can overflow in fp16: use load_4bit or fp32 for it.
        kw["torch_dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(name, device_map="auto", **kw).eval()
    return tok, model


def eos_ids(tok, model) -> List[int]:
    ids = set()
    g = getattr(model.generation_config, "eos_token_id", None)
    if isinstance(g, int):
        ids.add(g)
    elif g:
        ids.update(int(x) for x in g)
    if tok.eos_token_id is not None:
        ids.add(int(tok.eos_token_id))
    for t in ("<|im_end|>", "<|eot_id|>", "<end_of_turn>"):
        i = tok.convert_tokens_to_ids(t)
        if isinstance(i, int) and i >= 0 and i != tok.unk_token_id:
            ids.add(i)
    return sorted(ids)


def render(tok, messages: List[dict]) -> str:
    """Apply the chat template; fold the system prompt into the user turn for templates that reject it (Gemma)."""
    try:
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        merged = [{"role": "user", "content": messages[0]["content"] + "\n\n" + messages[1]["content"]}]
        return tok.apply_chat_template(merged, tokenize=False, add_generation_prompt=True)


def make_generate_fn(tok, model, max_new_tokens: int = 32, batch_size: int = 16):
    import torch

    stops = set(eos_ids(tok, model))

    def generate(prompts: List[List[dict]]) -> List[Tuple[str, bool]]:
        texts = [render(tok, m) for m in prompts]
        lens = [len(tok(t, add_special_tokens=False).input_ids) for t in texts]
        order = sorted(range(len(texts)), key=lambda i: lens[i])  # length-sorted batches: less padding
        results: List[Tuple[str, bool]] = [("", False)] * len(texts)
        i, bs = 0, batch_size
        while i < len(order):
            idx = order[i : i + bs]
            enc = tok([texts[j] for j in idx], return_tensors="pt", padding=True,
                      add_special_tokens=False).to(model.device)
            try:
                with torch.no_grad():
                    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                         eos_token_id=sorted(stops), pad_token_id=tok.pad_token_id)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)  # retry the same slice with a smaller batch
                continue
            new = out[:, enc.input_ids.shape[1]:]
            for row, j in zip(new, idx):
                ids = row.tolist()
                hit_eos = any(t in stops for t in ids)
                results[j] = (tok.decode(ids, skip_special_tokens=True), not hit_eos)
            i += len(idx)
        return results

    return generate

Writing slmrag_hf.py


In [6]:
%%writefile run_step1.py
"""Step 1 runner: baseline matrix (closed-book / oracle / BM25 / dense / hybrid) for one model + dataset.

  python run_step1.py --dataset hotpot --model Qwen/Qwen2.5-1.5B-Instruct --n 500 --smoke   # 20-question check
  python run_step1.py --dataset hotpot --model Qwen/Qwen2.5-1.5B-Instruct --n 1000
  python run_step1.py --dataset pubmed --model Qwen/Qwen2.5-1.5B-Instruct --settings closed,oracle,bm25

Writes per-example JSONL (RAW output kept) + summary JSON per setting into --out.
"""
from __future__ import annotations

import argparse
import json
import os
import re
import sys

import slmrag_core as C
import slmrag_data as D


def slug(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "-", s).strip("-")


def build_retrievers(settings, pool, args):
    texts = [p["text"] for p in pool]
    R = {}
    need_bm25 = {"bm25", "hybrid"} & set(settings)
    need_dense = {"dense", "hybrid"} & set(settings)
    if need_bm25:
        print(f"indexing BM25 over {len(texts):,} passages ...", flush=True)
        R["bm25"] = C.BM25(texts)
    if need_dense:
        print(f"encoding {len(texts):,} passages with {args.dense_model} ...", flush=True)
        enc = C.make_st_encoder(args.dense_model)
        qp, dp = ("query: ", "passage: ") if "e5" in args.dense_model.lower() else ("", "")
        R["dense"] = C.DenseRetriever(texts, enc, query_prefix=qp, doc_prefix=dp)
    if "hybrid" in settings:
        R["hybrid"] = C.HybridRRF(R["bm25"], R["dense"])
    return R, texts


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", choices=["nq", "hotpot", "pubmed"], required=True)
    ap.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    ap.add_argument("--settings", default="closed,oracle,bm25,dense,hybrid")
    ap.add_argument("--n", type=int, default=500)
    ap.add_argument("--k", type=int, default=5, help="passages placed in the prompt")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--max_new_tokens", type=int, default=32)
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--context_budget", type=int, default=600, help="max context words, identical for every retriever")
    ap.add_argument("--dense_model", default="intfloat/e5-base-v2")
    ap.add_argument("--corpus", default=None, help="JSONL of {id,text}; overrides the per-dataset pool (needed for NQ)")
    ap.add_argument("--load_4bit", action="store_true")
    ap.add_argument("--smoke", action="store_true", help="20 questions, closed+oracle only")
    ap.add_argument("--out", default="runs")
    args = ap.parse_args()

    settings = args.settings.split(",")
    if args.smoke:
        args.n, settings = 20, ["closed", "oracle"]
    os.makedirs(args.out, exist_ok=True)

    # data
    if args.dataset == "nq":
        recs, pool = D.load_nq(args.n, args.seed)
    elif args.dataset == "hotpot":
        recs, pool = D.load_hotpot(args.n, args.seed)
    else:
        recs, pool, _ = D.load_pubmed(args.n, args.seed)
    if args.corpus:
        pool = D.load_corpus_jsonl(args.corpus)
    gold_free = all(not r["gold_passages"] for r in recs)
    if pool is None:
        keep = [s for s in settings if s == "closed"]
        if set(settings) - set(keep):
            print("NQ has no built-in retrieval pool or gold passage; running closed-book only. "
                  "Pass --corpus (Step 2) for bm25/dense/hybrid.", file=sys.stderr)
        settings = keep
    if "oracle" in settings and gold_free:
        print("oracle skipped: dataset has no gold passages.", file=sys.stderr)
        settings = [s for s in settings if s != "oracle"]

    # model
    from slmrag_hf import load_model, make_generate_fn
    tok, model = load_model(args.model, args.load_4bit)
    gen = make_generate_fn(tok, model, args.max_new_tokens, args.batch_size)

    R, texts = build_retrievers(settings, pool, args) if pool else ({}, [])
    summaries = {}
    for s in settings:
        cache, ids = None, None
        if s == "oracle":
            cache = {r["id"]: r["gold_passages"] for r in recs}
        elif s != "closed":
            cache = {r["id"]: [texts[i] for i, _ in R[s].search(r["question"], max(10, args.k))] for r in recs}
        retrieve_fn = (lambda r, c=cache: c[r["id"]][: args.k]) if cache is not None else None

        rows, summ = C.evaluate(recs, gen, retrieve_fn, context_budget=args.context_budget)
        if cache is not None and s != "closed" and s != "oracle" and args.dataset != "pubmed":
            summ.update({f"ret_{k}": v for k, v in C.retrieval_metrics(
                [cache[r["id"]] for r in recs], [r["answers"] for r in recs]).items()})
        summaries[s] = summ
        base = os.path.join(args.out, f"{args.dataset}__{slug(args.model)}__{s}_k{args.k}")
        with open(base + ".jsonl", "w") as f:
            for row in rows:
                f.write(json.dumps(row) + "\n")
        with open(base + ".summary.json", "w") as f:
            json.dump({"args": vars(args), "setting": s, **summ}, f, indent=1)
        key = "yn_accuracy" if args.dataset == "pubmed" else "acc"
        print(f"{s:7s} n={summ['n']} {key}={summ.get(key, float('nan')):.3f} "
              f"f1={summ.get('f1', float('nan')):.3f} (raw f1 {summ.get('raw_f1', float('nan')):.3f}) "
              f"words={summ.get('mean_pred_words', float('nan')):.1f} hit_max={summ['hit_max_rate']:.1%}", flush=True)

    for w in C.sanity_warnings(summaries):
        print("WARNING:", w)
    with open(os.path.join(args.out, f"{args.dataset}__{slug(args.model)}__ALL.json"), "w") as f:
        json.dump(summaries, f, indent=1)


if __name__ == "__main__":
    main()

Writing run_step1.py


In [7]:
%%writefile compare_runs.py
"""Paired comparison of runs written by run_step1.py (aligned by example id, Holm-corrected).

  python compare_runs.py runs/hotpot__*__bm25_k5.jsonl --baseline runs/hotpot__*__closed_k5.jsonl --metric acc
  python compare_runs.py runs/*__oracle_k5.jsonl runs/*__bm25_k5.jsonl --baseline runs/*__closed_k5.jsonl --metric f1

Reports mean difference with a bootstrap CI, paired t / Wilcoxon, exact McNemar for 0/1 metrics,
Cohen's dz, Holm-adjusted p across all comparisons shown, and the minimum detectable effect so a
"not significant" result can be read as 'underpowered' or 'truly small'.
"""
from __future__ import annotations

import argparse
import json

import slmrag_core as C


def load(path, metric):
    with open(path) as f:
        return {r["id"]: float(r[metric]) for r in map(json.loads, f) if metric in r}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("runs", nargs="+")
    ap.add_argument("--baseline", required=True)
    ap.add_argument("--metric", default="acc", choices=["acc", "f1", "em"])
    args = ap.parse_args()

    base = load(args.baseline, args.metric)
    results = []
    for p in args.runs:
        cur = load(p, args.metric)
        ids = sorted(set(base) & set(cur))
        if not ids:
            print(f"{p}: no overlapping ids"); continue
        a, b = [cur[i] for i in ids], [base[i] for i in ids]
        r = C.paired_compare(a, b, binary=args.metric != "f1")
        r["mde"] = C.mde_paired(r["n"], r["sd_diff"]) if r["sd_diff"] > 0 else float("nan")
        results.append((p, r))

    key = "p_mcnemar" if args.metric != "f1" else "p_wilcoxon"
    adj = C.holm([r[key] for _, r in results]) if results else []
    print(f"metric={args.metric}  baseline={args.baseline}  (primary test: {key}, Holm over {len(results)} comparisons)")
    for (p, r), pa in zip(results, adj):
        print(f"{p}\n   n={r['n']}  diff={r['mean_diff']:+.4f} [{r['ci_lo']:+.4f},{r['ci_hi']:+.4f}]  "
              f"dz={r['dz']:+.2f}  p_t={r['p_t']:.3g}  p_wilcoxon={r['p_wilcoxon']:.3g}"
              + (f"  p_mcnemar={r['p_mcnemar']:.3g}" if "p_mcnemar" in r else "")
              + f"  Holm p={pa:.3g}  MDE(80%)={r['mde']:.4f}")


if __name__ == "__main__":
    main()

Writing compare_runs.py


In [8]:
%%writefile diagnose_floor.py
"""Confirm WHY F1 is ~0.04 using predictions you already have (no GPU needed).

  python diagnose_floor.py preds.csv --pred-col prediction --gold-col answer [--group-col domain]

Accepts CSV or JSONL. Gold may be a plain string, a JSON list, or 'a ||| b' aliases.
Reads: raw F1 and EM as you currently compute them, how long the outputs are, how often the
model went on to invent follow-up questions, whether the gold answer is CONTAINED in the raw
output, and the F1 after cutting the output at the first newline / invented question.

Interpretation: if containment accuracy is many times higher than raw F1, and cleaned F1
jumps, the model often knows the answer and the metric is punishing verbosity (the missing-EOS
problem in Section 6.7), i.e. the floor is a measurement artefact, not a capability result.
"""
from __future__ import annotations

import argparse
import json
import re

import numpy as np
import pandas as pd

import slmrag_core as C


def parse_gold(v):
    if isinstance(v, list):
        return [str(x) for x in v]
    s = str(v)
    if s.startswith("["):
        try:
            return [str(x) for x in json.loads(s)]
        except Exception:
            pass
    return [x.strip() for x in s.split("|||")]


def report(df, pred_col, gold_col, label="ALL"):
    preds = df[pred_col].fillna("").astype(str).tolist()
    golds = [parse_gold(g) for g in df[gold_col]]
    cleaned = [C.clean_generation(p) for p in preds]
    f1_raw = np.mean([C.f1_score(p, g) for p, g in zip(preds, golds)])
    em_raw = np.mean([C.exact_match(p, g) for p, g in zip(preds, golds)])
    acc_raw = np.mean([C.acc_contain(p, g) for p, g in zip(preds, golds)])
    f1_cl = np.mean([C.f1_score(p, g) for p, g in zip(cleaned, golds)])
    acc_cl = np.mean([C.acc_contain(p, g) for p, g in zip(cleaned, golds)])
    words = np.mean([len(p.split()) for p in preds])
    ramble = np.mean([bool(re.search(r"(\n|Question\s*:|Q\s*:)", p.strip(), flags=re.I)) for p in preds])
    print(f"[{label}] n={len(df)}")
    print(f"  as you score now : F1={f1_raw:.4f}  EM={em_raw:.4f}")
    print(f"  mean raw length  : {words:.1f} words;  {ramble:.0%} contain a newline or an invented 'Question:'")
    print(f"  gold CONTAINED in raw output : {acc_raw:.1%}")
    print(f"  after cutting at first line  : F1={f1_cl:.4f}  containment={acc_cl:.1%}")
    ratio = acc_raw / max(f1_raw, 1e-9)
    if ratio > 3 and f1_cl > 2 * f1_raw:
        print("  VERDICT: verbosity artefact. The answer is often present but buried; fix stop handling/EOS and re-run.")
    elif acc_raw < 0.05:
        print("  VERDICT: answers are mostly absent even in the raw text: check prompt, context, or the gold field you pass in.")
    else:
        print("  VERDICT: mixed; inspect examples below.")
    for p, g in list(zip(preds, golds))[:3]:
        print(f"   gold={g[:2]}  raw={p[:160]!r}")
    print()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("file")
    ap.add_argument("--pred-col", default="prediction")
    ap.add_argument("--gold-col", default="answer")
    ap.add_argument("--group-col", default=None)
    a = ap.parse_args()
    df = pd.read_json(a.file, lines=True) if a.file.endswith((".jsonl", ".json")) else pd.read_csv(a.file)
    report(df, a.pred_col, a.gold_col)
    if a.group_col:
        for g, sub in df.groupby(a.group_col):
            report(sub, a.pred_col, a.gold_col, str(g))


if __name__ == "__main__":
    main()

Writing diagnose_floor.py


In [9]:
%%writefile test_core.py
import numpy as np
import pytest

import slmrag_core as C


# ---- metrics -------------------------------------------------------------
def test_normalize():
    assert C.normalize_answer("The  Eiffel Tower!") == "eiffel tower"


def test_em_f1_aliases():
    golds = ["Barack Obama", "Obama"]
    assert C.exact_match("obama", golds) == 1.0
    assert C.f1_score("President Barack Obama", golds) == pytest.approx(0.8)  # 2*(2/3*1)/(2/3+1)
    assert C.f1_score("Paris", golds) == 0.0


def test_contains_token_boundary():
    assert C.contains_answer("He was born in Paris, France.", ["Paris"])
    assert not C.contains_answer("The party was fun", ["art"])
    assert not C.contains_answer("anything", [""])


def test_label_extraction_and_metrics():
    assert C.extract_label("Yes.") == "yes"
    assert C.extract_label("I am not sure") is None
    m = C.label_metrics(["yes", "no", None, "maybe"], ["yes", "no", "yes", "no"])
    assert m["accuracy"] == 0.5
    assert 0 <= m["macro_f1"] <= 1


# ---- generation cleaning (the root cause of the F1 floor) ----------------
@pytest.mark.parametrize("raw,expected", [
    ("Paris\nQuestion: What is the capital of Spain?\nAnswer: Madrid", "Paris"),
    ("Answer: Barack Obama.", "Barack Obama"),
    ("  \"Paris\"  ", "Paris"),
    ("Paris Question: who wrote Hamlet? Answer: Shakespeare", "Paris"),
    ("Final answer: 1969", "1969"),
    ("Franklin D. Roosevelt", "Franklin D. Roosevelt"),
])
def test_clean_generation(raw, expected):
    assert C.clean_generation(raw) == expected


def test_clean_generation_max_words():
    assert C.clean_generation("one two three four", max_words=2) == "one two"


def test_rambling_output_f1_floor_reproduced():
    """A correct answer followed by fabricated Q/A pairs scores near zero raw, ~1 cleaned."""
    raw = "Paris\n" + "\n".join(f"Question: what is item {i}?\nAnswer: thing number {i} is here" for i in range(6))
    assert C.f1_score(raw, "Paris") < 0.1
    assert C.f1_score(C.clean_generation(raw), "Paris") == 1.0


# ---- prompts -------------------------------------------------------------
def test_truncate_passages_keeps_whole_passages():
    ps = ["a b c", "d e f", "g h i"]
    assert C.truncate_passages(ps, 6) == ["a b c", "d e f"]
    assert C.truncate_passages(["a b c d e"], 3) == ["a b c"]


def test_build_messages_closed_vs_rag():
    closed = C.build_messages("Who?", None)
    rag = C.build_messages("Who?", ["p1", "p2"])
    assert "Context" not in closed[1]["content"]
    assert "[2] p2" in rag[1]["content"]
    assert "yes, no, or maybe" in C.build_messages("q", None, "yesno")[0]["content"]


# ---- retrieval -----------------------------------------------------------
DOCS = [
    "Paris is the capital of France and a major European city",
    "Madrid is the capital of Spain",
    "The mitochondria is the powerhouse of the cell",
    "Berlin is the capital of Germany",
]


def test_bm25_ranks_relevant_first():
    bm = C.BM25(DOCS)
    assert bm.search("capital of France", 2)[0][0] == 0
    assert bm.search("mitochondria cell", 1)[0][0] == 2
    assert bm.search("zzzz unknown", 2) is not None  # no crash on OOV


def test_dense_with_fake_encoder():
    vocab = {w: i for i, w in enumerate(sorted({t for d in DOCS for t in C.tokenize(d)} | {"france"}))}

    def enc(texts):
        out = np.zeros((len(texts), len(vocab)))
        for i, t in enumerate(texts):
            for w in C.tokenize(t):
                if w in vocab:
                    out[i, vocab[w]] += 1
        return out

    d = C.DenseRetriever(DOCS, enc)
    assert d.search("capital france", 1)[0][0] == 0


def test_rrf_and_hybrid():
    fused = C.rrf([[1, 2, 3], [3, 1, 4]], top=3)
    assert fused[0][0] == 1
    class A:  # stub retrievers
        def search(self, q, k): return [(0, 1.0), (1, 0.5)]
    class B:
        def search(self, q, k): return [(1, 1.0), (0, 0.5)]
    assert len(C.HybridRRF(A(), B()).search("x", 2)) == 2


def test_retrieval_metrics():
    ret = [["no", "Paris is here", "x"], ["nothing", "nothing", "nothing"]]
    m = C.retrieval_metrics(ret, [["Paris"], ["Rome"]], ks=(1, 3))
    assert m["hit@1"] == 0.0 and m["hit@3"] == 0.5 and m["mrr@3"] == pytest.approx(0.25)


def test_chunking():
    assert C.chunk_words("a b c d e", 2, 0) == ["a b", "c d", "e"]
    assert C.chunk_words("", 4) == []


def test_id_recall():
    assert C.id_recall_at_k([[1, 2, 3]], [[1, 9]], 3) == 0.5


# ---- statistics ----------------------------------------------------------
def test_bootstrap_ci_covers_mean():
    x = np.random.default_rng(1).normal(0.5, 0.2, 400)
    m, lo, hi = C.bootstrap_ci(x, B=2000)
    assert lo < m < hi and hi - lo < 0.1


def test_paired_compare_detects_shift_and_null():
    rng = np.random.default_rng(0)
    a = rng.normal(0, 1, 300)
    out = C.paired_compare(a + 0.3, a + rng.normal(0, 0.3, 300), B=1000)
    assert out["p_t"] < 0.001 and out["ci_lo"] > 0
    null = C.paired_compare(a, a + rng.normal(0, 0.3, 300), B=1000)
    assert null["p_t"] > 0.01


def test_paired_compare_identical_inputs():
    x = np.ones(50)
    out = C.paired_compare(x, x, binary=True, B=200)
    assert out["p_t"] == 1.0 and out["p_wilcoxon"] == 1.0 and out["p_mcnemar"] == 1.0


def test_mcnemar_exact():
    a = np.array([1] * 20 + [0] * 5 + [1] * 10)
    b = np.array([0] * 20 + [1] * 5 + [1] * 10)
    out = C.paired_compare(a, b, binary=True, B=200)
    assert out["discordant"] == 25 and out["p_mcnemar"] < 0.01


def test_holm_monotone_and_bounded():
    adj = C.holm([0.01, 0.04, 0.03, 0.5])
    assert adj[0] == pytest.approx(0.04) and max(adj) <= 1.0
    assert adj[1] >= adj[2] >= adj[0]


def test_power_matches_manuscript_headline():
    """Manuscript: dz=0.124, diff=0.0059 => SD(diff)~0.0476; at n=300 the MDE exceeds the observed effect."""
    sd = 0.0059 / 0.124
    assert C.mde_paired(300, sd) > 0.0059
    assert C.n_required(0.0059, sd) > 300


def test_kendall():
    tau, p = C.kendall_tau([1, 2, 3, 4], [1, 2, 3, 4])
    assert tau == pytest.approx(1.0)


# ---- evaluate() end to end with a fake rambling model ---------------------
def _records():
    return [
        {"id": "1", "question": "Capital of France?", "answers": ["Paris"], "task": "qa"},
        {"id": "2", "question": "Capital of Spain?", "answers": ["Madrid"], "task": "qa"},
    ]


def test_evaluate_cleaning_and_summary():
    def fake_gen(prompts):
        outs = []
        for p in prompts:
            ans = "Paris" if "Question: Capital of France" in p[1]["content"] else "Madrid"
            outs.append((ans + "\nQuestion: next?\nAnswer: blah blah blah blah", False))
        return outs

    rows, s = C.evaluate(_records(), fake_gen, retrieve_fn=lambda r: DOCS[:2])
    assert s["f1"] == 1.0 and s["acc"] == 1.0 and s["raw_f1"] < 0.5
    assert rows[0]["ctx_has_answer"] is True
    assert any("post-processing" in w for w in C.sanity_warnings({"x": s}))


def test_evaluate_yesno():
    recs = [{"id": "p1", "question": "q?", "label": "yes", "task": "yesno"},
            {"id": "p2", "question": "q?", "label": "no", "task": "yesno"}]
    rows, s = C.evaluate(recs, lambda ps: [("Yes.", False), ("maybe", False)])
    assert s["yn_accuracy"] == 0.5 and s["yn_unparsed"] == 0.0


def test_sanity_flags_runaway_and_oracle_regression():
    good = {"acc": 0.4, "hit_max_rate": 0.0, "empty_rate": 0.0, "mean_pred_words": 3}
    bad = {"acc": 0.3, "hit_max_rate": 0.4, "empty_rate": 0.0, "mean_pred_words": 20}
    w = C.sanity_warnings({"closed": good, "oracle": bad})
    assert any("max_new_tokens" in x for x in w)
    assert any("oracle" in x for x in w)


# ---- data transforms (schema assumed from the HF dataset cards; verify with `slmrag_data.py --check`) ----
import slmrag_data as D


def test_hotpot_record_gold_passages():
    row = {"id": "abc", "question": "Who?", "answer": "Bob",
           "context": {"title": ["A", "B", "C"], "sentences": [["a1.", "a2."], ["b1."], ["c1."]]},
           "supporting_facts": {"title": ["B", "A", "B"], "sent_id": [0, 1, 0]}}
    r = D.hotpot_record(row)
    assert r["gold_ids"] == ["B", "A"] and r["gold_passages"][0].startswith("B: b1.")
    assert r["answers"] == ["Bob"] and r["task"] == "qa"


def test_pubmed_and_nq_records():
    pm = D.pubmed_record({"pubid": 7, "question": "Does X?", "final_decision": "Maybe",
                          "context": {"contexts": ["one.", "two."]}})
    assert pm["label"] == "maybe" and pm["gold_passages"] == ["one. two."] and pm["task"] == "yesno"
    nq = D.nq_record(3, {"question": "who sings x", "answer": ["Reba", "Reba McEntire"]})
    assert nq["question"] == "who sings x?" and len(nq["answers"]) == 2


# ---- runner end-to-end with a fake model and fake data (no torch / no network) ----
def test_run_step1_end_to_end(tmp_path, monkeypatch, capsys):
    import sys, types, json, run_step1

    recs = [{"id": f"hp-{i}", "question": f"Capital of C{i}?", "task": "qa", "answers": [f"Town{i}"],
             "gold_passages": [f"C{i}: the capital of C{i} is Town{i}."], "gold_ids": [f"C{i}"]} for i in range(30)]
    pool = [{"id": f"C{i}", "text": f"C{i}: the capital of C{i} is Town{i}."} for i in range(30)]
    monkeypatch.setattr(run_step1.D, "load_hotpot", lambda n, seed: (recs[:n], pool))

    def fake_gen(prompts):
        outs = []
        for p in prompts:
            c = p[1]["content"]
            import re as _re
            q = _re.search(r"Question: Capital of (C\d+)\?", c).group(1)
            ans = f"Town{q[1:]}" if f"capital of {q} is" in c else "unknown"
            outs.append((ans + "\nQuestion: more?\nAnswer: filler filler", False))
        return outs

    fake = types.ModuleType("slmrag_hf")
    fake.load_model = lambda name, load_4bit=False: (None, None)
    fake.make_generate_fn = lambda tok, model, mnt, bs: fake_gen
    monkeypatch.setitem(sys.modules, "slmrag_hf", fake)
    monkeypatch.setattr(sys, "argv", ["run_step1.py", "--dataset", "hotpot", "--n", "30", "--k", "3",
                                      "--settings", "closed,oracle,bm25", "--out", str(tmp_path)])
    run_step1.main()
    out = capsys.readouterr().out
    assert "closed" in out and "oracle" in out and "bm25" in out
    allj = json.load(open(next(tmp_path.glob("*__ALL.json"))))
    assert allj["closed"]["acc"] == 0.0 and allj["oracle"]["acc"] == 1.0 and allj["bm25"]["acc"] == 1.0
    assert allj["oracle"]["f1"] == 1.0 and allj["oracle"]["raw_f1"] < 0.5
    assert allj["bm25"]["ret_hit@5"] == 1.0
    rows = [json.loads(l) for l in open(next(tmp_path.glob("*__oracle_k3.jsonl")))]
    assert "Question: more?" in rows[0]["raw"]  # raw output retained for audit


def test_compare_runs_and_diagnose(tmp_path, monkeypatch, capsys):
    import sys, json, compare_runs, diagnose_floor, pandas as pd
    for name, accs in (("a", [1] * 40 + [0] * 10), ("b", [0] * 25 + [1] * 25)):
        with open(tmp_path / f"{name}.jsonl", "w") as f:
            for i, v in enumerate(accs):
                f.write(json.dumps({"id": str(i), "acc": v}) + "\n")
    monkeypatch.setattr(sys, "argv", ["c", str(tmp_path / "a.jsonl"), "--baseline", str(tmp_path / "b.jsonl")])
    compare_runs.main()
    assert "Holm p=" in capsys.readouterr().out

    df = pd.DataFrame({"prediction": ["Paris\nQuestion: x?\nAnswer: y y y y y y y y"] * 10,
                       "answer": ["Paris"] * 10})
    diagnose_floor.report(df, "prediction", "answer")
    assert "verbosity artefact" in capsys.readouterr().out

Writing test_core.py


## 2. Tests and schema check

In [10]:
!python -m pytest -q -p no:cacheprovider test_core.py

..................................                                       [100%]
34 passed in 1.26s


In [11]:
# Downloads 3 datasets (1-2 min). Fails loudly if a Hugging Face schema differs from what the loaders expect.
!python slmrag_data.py --check

README.md: 100%|███████████████████████████| 8.77k/8.77k [00:00<00:00, 24.7MB/s]

nq_open/train-00000-of-00001.parquet: downloading bytes: █▍ | 2.10MB            
nq_open/train-00000-of-00001.parquet: downloading bytes: ███| 4.44MB,  435kB/s  
nq_open/train-00000-of-00001.parquet: reconstructing file: 100%|█| 4.46MB / 4.46

nq_open/validation-00000-of-00001.parque(…): downloading bytes:  |  0.00B       
nq_open/validation-00000-of-00001.parque(…): downloading bytes: █|  212kB, 21.0k
nq_open/validation-00000-of-00001.parque(…): reconstructing file: 100%|█|  214kB
Generating train split: 100%|██| 87925/87925 [00:00<00:00, 723584.37 examples/s]
Generating validation split: 100%|█| 3610/3610 [00:00<00:00, 794826.11 examples/
README.md: 100%|███████████████████████████| 9.52k/9.52k [00:00<00:00, 16.7MB/s]

distractor/train-00000-of-00002.parquet: downloading bytes:  |  0.00B           
distractor/train-00000-of-00002.parquet: downloading bytes: ▏| 34.5MB, 1.60MB/s 
distractor/train-00000-of

## 3. (Optional) Diagnose your old predictions

Upload your old per-example predictions as a Kaggle dataset (CSV or JSONL), set the path and column names, and run.
The verdict tells you whether the old F1 of about 0.04 was a verbosity artefact. Skipped automatically if the file is missing.

In [12]:
OLD_PREDS = "/kaggle/input/YOUR_DATASET/old_predictions.csv"   # <- edit
PRED_COL, GOLD_COL, GROUP_COL = "prediction", "answer", "domain"  # <- edit (GROUP_COL = "" to skip)

if os.path.exists(OLD_PREDS):
    g = f"--group-col {GROUP_COL}" if GROUP_COL else ""
    !python diagnose_floor.py {OLD_PREDS} --pred-col {PRED_COL} --gold-col {GOLD_COL} {g}
else:
    print("No old-predictions file found; skipping (edit OLD_PREDS to use this step).")

No old-predictions file found; skipping (edit OLD_PREDS to use this step).


## 4. Configuration

In [13]:
MODEL      = "Qwen/Qwen2.5-1.5B-Instruct"
N_HOTPOT   = 1000        # eval questions; n >= 1000 so a ~0.004 effect is detectable
N_PUBMED   = 500         # pqa_labeled has only 1000 labelled rows; 500 kept disjoint for later fine-tuning
N_NQ       = 1000
K          = 5           # passages placed in the prompt
SEED       = 42
LOAD_4BIT  = False       # fp16 fits a 1.5B model on a T4; set True for 3B+ or Gemma-2
OUT        = "/kaggle/working/runs"

Q4 = "--load_4bit" if LOAD_4BIT else ""
from run_step1 import slug
MSLUG = slug(MODEL)
print(MODEL, "->", MSLUG)

Qwen/Qwen2.5-1.5B-Instruct -> Qwen-Qwen2-5-1-5B-Instruct


## 5. Smoke test (20 questions, closed-book + oracle)

In [14]:
!python run_step1.py --dataset hotpot --model {MODEL} --smoke {Q4} --out {OUT}/smoke

config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 3.21MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 19.1MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 44.2MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 118MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 158MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:                       |  0.00B            
model.safetensors: downloading bytes: ▍                     | 53.9MB, 2.93MB/s  
model.safetensors: downloading bytes: █▎                    |  186MB, 13.5MB/s  
model.safetensors: downloading bytes: █▋                    |  241MB, 18.6MB/s  
model.safetensors: downloading bytes: █▉                    |  277MB, 22.5MB/s  
model.safetensors: downloading bytes: ███▉                  |  555MB, 44.4MB/s  
model.safetensors: downloading bytes: ████▎

Read the output: `hit_max` should be **below 2%**, mean answer length **at most 8 words**, and **no WARNING lines**.
If a WARNING appears, stop and look at the raw text:

In [15]:
import json, glob
for f in sorted(glob.glob(f"{OUT}/smoke/*oracle*.jsonl"))[:1]:
    rows = [json.loads(l) for l in open(f)]
    for r in rows[:5]:
        print("GOLD :", r["answers"][:2]); print("RAW  :", repr(r["raw"][:200])); print("CLEAN:", repr(r["pred"])); print("-"*60)

GOLD : ['flotilla']
RAW  : 'Vice-Admiral'
CLEAN: 'Vice-Admiral'
------------------------------------------------------------
GOLD : ['15 August 1843']
RAW  : '1843'
CLEAN: '1843'
------------------------------------------------------------
GOLD : ['Armenian']
RAW  : 'Armenia'
CLEAN: 'Armenia'
------------------------------------------------------------
GOLD : ['Centre of Excellence']
RAW  : 'Centre of Excellence'
CLEAN: 'Centre of Excellence'
------------------------------------------------------------
GOLD : ['Saoirse Ronan']
RAW  : 'Saoirse Ronan'
CLEAN: 'Saoirse Ronan'
------------------------------------------------------------


## 6. Baseline matrix

In [16]:
# HotpotQA (closed pool: all paragraphs of the validation split). ~5 settings.
!python run_step1.py --dataset hotpot --model {MODEL} --n {N_HOTPOT} --k {K} --seed {SEED} {Q4} --out {OUT}

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 247.24it/s]
indexing BM25 over 66,635 passages ...
encoding 66,635 passages with intfloat/e5-base-v2 ...
modules.json: 100%|████████████████████████████| 387/387 [00:00<00:00, 2.04MB/s]
README.md: 100%|████████████████████████████| 67.6k/67.6k [00:00<00:00, 102MB/s]
sentence_bert_config.json: 100%|██████████████| 57.0/57.0 [00:00<00:00, 221kB/s]
config.json: 100%|█████████████████████████████| 650/650 [00:00<00:00, 2.03MB/s]

model.safetensors: downloading bytes:                       |  0.00B            
model.safetensors: downloading bytes: ██▍                   | 49.5MB,   ???B/s  
model.safetensors: downloading bytes: █████▋                |  114MB, 7.91MB/s  
model.safetensors: downloading bytes: ███████▍              |  147MB, 11.1MB/s  
model.safetensors: downloading bytes: ████████████▎         |  246MB, 20.3MB/s  
model.safetensors: downloading 

In [17]:
# PubMedQA, scored as yes/no/maybe classification
!python run_step1.py --dataset pubmed --model {MODEL} --n {N_PUBMED} --k 3 --seed {SEED} {Q4} --out {OUT}

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 273.58it/s]
indexing BM25 over 1,000 passages ...
encoding 1,000 passages with intfloat/e5-base-v2 ...
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2693.26it/s]
closed  n=500 yn_accuracy=0.578 f1=nan (raw f1 nan) words=nan hit_max=0.0%
oracle  n=500 yn_accuracy=0.652 f1=nan (raw f1 nan) words=nan hit_max=0.0%
bm25    n=500 yn_accuracy=0.638 f1=nan (raw f1 nan) words=nan hit_max=0.0%
dense   n=500 yn_accuracy=0.638 f1=nan (raw f1 nan) words=nan hit_max=0.0%
hybrid  n=500 yn_accuracy=0.642 f1=nan (raw f1 nan) words=nan hit_max=0.0%


In [18]:
# NQ-Open: closed-book only until Step 2 provides a Wikipedia corpus
!python run_step1.py --dataset nq --model {MODEL} --n {N_NQ} --seed {SEED} --settings closed {Q4} --out {OUT}

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:01<00:00, 264.02it/s]
closed  n=1000 acc=0.152 f1=0.168 (raw f1 0.168) words=2.4 hit_max=0.1%


## 7. Paired statistics (aligned by example id, Holm-corrected)

In [19]:
print("=== HotpotQA: containment accuracy vs closed-book ===")
!python compare_runs.py {OUT}/hotpot__{MSLUG}__oracle_k{K}.jsonl {OUT}/hotpot__{MSLUG}__bm25_k{K}.jsonl {OUT}/hotpot__{MSLUG}__dense_k{K}.jsonl {OUT}/hotpot__{MSLUG}__hybrid_k{K}.jsonl --baseline {OUT}/hotpot__{MSLUG}__closed_k{K}.jsonl --metric acc
print()
print("=== HotpotQA: F1 vs closed-book ===")
!python compare_runs.py {OUT}/hotpot__{MSLUG}__oracle_k{K}.jsonl {OUT}/hotpot__{MSLUG}__bm25_k{K}.jsonl {OUT}/hotpot__{MSLUG}__dense_k{K}.jsonl {OUT}/hotpot__{MSLUG}__hybrid_k{K}.jsonl --baseline {OUT}/hotpot__{MSLUG}__closed_k{K}.jsonl --metric f1
print()
print("=== HotpotQA: BM25 vs dense vs hybrid (head to head, baseline = BM25) ===")
!python compare_runs.py {OUT}/hotpot__{MSLUG}__dense_k{K}.jsonl {OUT}/hotpot__{MSLUG}__hybrid_k{K}.jsonl --baseline {OUT}/hotpot__{MSLUG}__bm25_k{K}.jsonl --metric acc

=== HotpotQA: containment accuracy vs closed-book ===
metric=acc  baseline=/kaggle/working/runs/hotpot__Qwen-Qwen2-5-1-5B-Instruct__closed_k5.jsonl  (primary test: p_mcnemar, Holm over 4 comparisons)
/kaggle/working/runs/hotpot__Qwen-Qwen2-5-1-5B-Instruct__oracle_k5.jsonl
   n=1000  diff=+0.4580 [+0.4240,+0.4920]  dz=+0.83  p_t=6.82e-116  p_wilcoxon=9.49e-91  p_mcnemar=4.95e-109  Holm p=1.98e-108  MDE(80%)=0.0489
/kaggle/working/runs/hotpot__Qwen-Qwen2-5-1-5B-Instruct__bm25_k5.jsonl
   n=1000  diff=+0.2300 [+0.2000,+0.2600]  dz=+0.48  p_t=2.17e-47  p_wilcoxon=5.45e-43  p_mcnemar=3.68e-49  Holm p=3.68e-49  MDE(80%)=0.0422
/kaggle/working/runs/hotpot__Qwen-Qwen2-5-1-5B-Instruct__dense_k5.jsonl
   n=1000  diff=+0.2710 [+0.2390,+0.3030]  dz=+0.52  p_t=2.93e-54  p_wilcoxon=1.74e-48  p_mcnemar=9.44e-55  Holm p=2.83e-54  MDE(80%)=0.0460
/kaggle/working/runs/hotpot__Qwen-Qwen2-5-1-5B-Instruct__hybrid_k5.jsonl
   n=1000  diff=+0.2610 [+0.2300,+0.2930]  dz=+0.51  p_t=5.72e-52  p_wilcoxon=1.13e-4

In [20]:
print("=== PubMedQA: correctness vs closed-book ===")
!python compare_runs.py {OUT}/pubmed__{MSLUG}__oracle_k3.jsonl {OUT}/pubmed__{MSLUG}__bm25_k3.jsonl {OUT}/pubmed__{MSLUG}__dense_k3.jsonl {OUT}/pubmed__{MSLUG}__hybrid_k3.jsonl --baseline {OUT}/pubmed__{MSLUG}__closed_k3.jsonl --metric acc

=== PubMedQA: correctness vs closed-book ===
metric=acc  baseline=/kaggle/working/runs/pubmed__Qwen-Qwen2-5-1-5B-Instruct__closed_k3.jsonl  (primary test: p_mcnemar, Holm over 4 comparisons)
/kaggle/working/runs/pubmed__Qwen-Qwen2-5-1-5B-Instruct__oracle_k3.jsonl
   n=500  diff=+0.0740 [+0.0400,+0.1080]  dz=+0.19  p_t=1.66e-05  p_wilcoxon=1.93e-05  p_mcnemar=2.24e-05  Holm p=8.97e-05  MDE(80%)=0.0477
/kaggle/working/runs/pubmed__Qwen-Qwen2-5-1-5B-Instruct__bm25_k3.jsonl
   n=500  diff=+0.0600 [+0.0260,+0.0940]  dz=+0.15  p_t=0.000646  p_wilcoxon=0.000682  p_mcnemar=0.000901  Holm p=0.00107  MDE(80%)=0.0490
/kaggle/working/runs/pubmed__Qwen-Qwen2-5-1-5B-Instruct__dense_k3.jsonl
   n=500  diff=+0.0600 [+0.0280,+0.0940]  dz=+0.16  p_t=0.000381  p_wilcoxon=0.000407  p_mcnemar=0.000535  Holm p=0.00107  MDE(80%)=0.0470
/kaggle/working/runs/pubmed__Qwen-Qwen2-5-1-5B-Instruct__hybrid_k3.jsonl
   n=500  diff=+0.0640 [+0.0320,+0.0960]  dz=+0.17  p_t=0.000119  p_wilcoxon=0.000131  p_mcnemar=0.000

## 8. Summary table

In [21]:
import json, glob, pandas as pd
frames = []
for f in sorted(glob.glob(f"{OUT}/*{MSLUG}__ALL.json")):
    ds = os.path.basename(f).split("__")[0]
    d = json.load(open(f))
    df = pd.DataFrame(d).T
    df.insert(0, "dataset", ds)
    frames.append(df)
cols = ["dataset","n","acc","acc_lo","acc_hi","em","f1","raw_f1","yn_accuracy","yn_macro_f1","yn_unparsed",
        "mean_pred_words","mean_raw_words","hit_max_rate","ret_hit@1","ret_hit@5","ret_hit@10"]
summary = pd.concat(frames)
summary = summary[[c for c in cols if c in summary.columns]].round(3)
summary.index.name = "setting"
summary.to_csv(f"{OUT}/step1_summary_{MSLUG}.csv")
summary

,dataset,n,acc,acc_lo,acc_hi,em,f1,raw_f1,yn_accuracy,yn_macro_f1,yn_unparsed,mean_pred_words,mean_raw_words,hit_max_rate,ret_hit@1,ret_hit@5,ret_hit@10
setting,,,,,,,,,,,,,,,,,
closed,hotpot,1000.0,0.135,0.115,0.157,0.105,0.173,0.173,NaN,NaN,NaN,2.275,2.275,0.000,NaN,NaN,NaN
oracle,hotpot,1000.0,0.593,0.562,0.625,0.509,0.682,0.682,NaN,NaN,NaN,2.262,2.262,0.000,NaN,NaN,NaN
bm25,hotpot,1000.0,0.365,0.337,0.396,0.316,0.435,0.435,NaN,NaN,NaN,2.261,2.261,0.000,0.333,0.576,0.662
dense,hotpot,1000.0,0.406,0.375,0.436,0.350,0.477,0.477,NaN,NaN,NaN,2.247,2.247,0.000,0.363,0.703,0.778
hybrid,hotpot,1000.0,0.396,0.367,0.425,0.337,0.470,0.470,NaN,NaN,NaN,2.273,2.273,0.000,0.354,0.677,0.761
closed,nq,1000.0,0.152,0.130,0.174,0.095,0.168,0.168,NaN,NaN,NaN,2.448,2.458,0.001,NaN,NaN,NaN
closed,pubmed,500.0,NaN,NaN,NaN,NaN,NaN,NaN,0.578,0.314,0.0,NaN,NaN,0.000,NaN,NaN,NaN
oracle,pubmed,500.0,NaN,NaN,NaN,NaN,NaN,NaN,0.652,0.385,0.0,NaN,NaN,0.000,NaN,NaN,NaN
bm25,pubmed,500.0,NaN,NaN,NaN,NaN,NaN,NaN,0.638,0.378,0.0,NaN,NaN,0.000,NaN,NaN,NaN


How to read it: `raw_f1` is F1 on the uncleaned generation (what the old pipeline effectively measured); `f1` is after cutting at the first line.
A large gap means the model was rambling. `ret_hit@k` is answer-containment retrieval quality (HotpotQA and NQ only).

## 9. Package the outputs

In [22]:
import shutil
shutil.make_archive("/kaggle/working/step1_outputs", "zip", OUT)
print("saved /kaggle/working/step1_outputs.zip  (also visible under the notebook's Output tab after 'Save Version')")

saved /kaggle/working/step1_outputs.zip  (also visible under the notebook's Output tab after 'Save Version')


## Gates before Step 2
- [ ] Smoke test: `hit_max` below 2%, mean answer at most 8 words, no WARNING lines
- [ ] HotpotQA and PubMedQA: oracle beats closed-book with a paired CI that excludes 0
- [ ] `raw_f1` vs `f1` gap understood (recorded in the paper's limitations)
- [ ] If oracle accuracy on HotpotQA is still in single digits, stop and send 10 raw outputs from `runs/*.jsonl` (field `raw`)

Then send me `step1_summary_*.csv` and I will build Step 2 (open-domain Wikipedia corpus, NQ + HotpotQA + PopQA/TriviaQA, 0.5B / 1.5B / 3B scale axis).